#Implement a program that detects redundant computations, dead code, and strength reduction.


In [2]:
import re

def optimize_code(code_lines: list[str]) -> list[str]:
    """
    Optimizes a simple sequence of arithmetic assignments by performing
    strength reduction, redundant computation elimination, and dead code elimination.

    Args:
        code_lines: A list of strings, where each string is a line of code.

    Returns:
        A list of strings representing the optimized code.
    """
    # Maps a variable to its simplified expression or a constant value
    assignments = {}
    # Preserves the original order of variable assignments
    order = []

    # Regex to parse lines like 'var = operand1 operator operand2'
    line_regex = re.compile(r"(\w+)\s*=\s*(\w+)\s*([+\-*/])\s*(\w+)")

    # --- PHASE 1: Strength Reduction and Redundant Computation Elimination ---
    print("\n🔬 Phase 1: Analyzing expressions...")
    for line in code_lines:
        match = line_regex.match(line.strip())
        if not match:
            continue

        var, op1, operator, op2 = match.groups()
        order.append(var)

        # Strength Reduction (Constant Folding): If both operands are numbers
        if op1.isdigit() and op2.isdigit():
            val1, val2 = int(op1), int(op2)
            if operator == '*': result = val1 * val2
            elif operator == '+': result = val1 + val2
            elif operator == '-': result = val1 - val2
            # Use integer division for simplicity
            elif operator == '/': result = val1 // val2
            assignments[var] = str(result)
            print(f"   - Strength Reduction: '{line}' -> '{var} = {result}'")
            continue

        # Redundant Computation (Identity Elimination)
        # Case 1: x = y * 1 or x = 1 * y -> x = y
        if operator == '*' and (op1 == '1' or op2 == '1'):
            result = op2 if op1 == '1' else op1
            assignments[var] = result
            print(f"   - Redundant Computation: '{line}' -> '{var} = {result}'")
            continue
        # Case 2: x = y + 0 or x = 0 + y -> x = y
        if operator == '+' and (op1 == '0' or op2 == '0'):
            result = op2 if op1 == '0' else op1
            assignments[var] = result
            print(f"   - Redundant Computation: '{line}' -> '{var} = {result}'")
            continue

        # If no optimization applied, store the original expression
        assignments[var] = f"{op1} {operator} {op2}"

    # --- PHASE 2: Copy Propagation ---
    print("\n🔄 Phase 2: Propagating copies...")
    # Repeatedly substitute variables until no more changes can be made
    changed = True
    while changed:
        changed = False
        for var, expr in assignments.items():
            # Find any variable used in the expression
            used_vars = re.findall(r'[a-zA-Z_]\w*', expr)
            for used_var in used_vars:
                # If the used variable is just a copy of another, substitute it
                if used_var in assignments and assignments[used_var].isidentifier():
                    new_expr = expr.replace(used_var, assignments[used_var])
                    if new_expr != assignments[var]:
                        print(f"   - Propagating '{used_var}' in '{var} = {expr}' -> '{var} = {new_expr}'")
                        assignments[var] = new_expr
                        changed = True

    # --- PHASE 3: Dead Code Elimination ---
    print("\n🗑️ Phase 3: Eliminating dead code...")
    live_vars = set()
    # Assume the last variable computed is essential for the output
    if order:
        # Start a queue with the last assigned variable
        queue = [order[-1]]
        live_vars.add(order[-1])

        head = 0
        while head < len(queue):
            current_var = queue[head]
            head += 1

            # Find variables that current_var depends on
            if current_var in assignments:
                expr = assignments[current_var]
                dependencies = re.findall(r'[a-zA-Z_]\w*', expr)
                for dep in dependencies:
                    if dep not in live_vars:
                        live_vars.add(dep)
                        queue.append(dep)

    print(f"   - Live variables identified: {sorted(list(live_vars))}")

    # --- Final Code Generation ---
    optimized_code = []
    for var in order:
        if var in live_vars:
            optimized_code.append(f"{var} = {assignments[var]}")

    return optimized_code


# --- Main Execution ---
if __name__ == "__main__":
    input_code = []
    print("Enter your code line by line (e.g., 'a = 10 + 5').")
    print("Press Enter on an empty line when you are finished.")
    print("-" * 20)

    # Loop to gather user input
    while True:
        line = input()
        if not line:
            break
        input_code.append(line)

    if not input_code:
        print("No code was entered. Exiting.")
    else:
        print("\n--- Input Code ---")
        for l in input_code:
            print(l)
        print("-" * 20)

        optimized_output = optimize_code(input_code)

        print("\n--- Output (After Optimization) ---")
        if not optimized_output:
            print("No code left after optimization.")
        else:
            for l in optimized_output:
                print(l)
        print("-" * 20)

Enter your code line by line (e.g., 'a = 10 + 5').
Press Enter on an empty line when you are finished.
--------------------
x=2*8
y=x*1
z=y+0


--- Input Code ---
x=2*8
y=x*1
z=y+0
--------------------

🔬 Phase 1: Analyzing expressions...
   - Strength Reduction: 'x=2*8' -> 'x = 16'
   - Redundant Computation: 'y=x*1' -> 'y = x'
   - Redundant Computation: 'z=y+0' -> 'z = y'

🔄 Phase 2: Propagating copies...
   - Propagating 'y' in 'z = y' -> 'z = x'

🗑️ Phase 3: Eliminating dead code...
   - Live variables identified: ['x', 'z']

--- Output (After Optimization) ---
x = 16
z = x
--------------------


#Implement a simple code generator that translates arithmetic expressions into target assembly for a stack machine.

In [3]:
def generate_stack_code(infix_expression: str) -> list[str]:
    """
    Generates assembly code for a stack machine from an infix arithmetic expression.

    Args:
        infix_expression: A string containing the arithmetic expression.

    Returns:
        A list of strings representing the assembly instructions.
    """
    # Define operator precedence and the mapping from operator to assembly instruction
    precedence = {'+': 1, '-': 1, '*': 2, '/': 2}
    op_to_instruction = {'+': 'ADD', '-': 'SUB', '*': 'MUL', '/': 'DIV'}

    # --- Step 1: Convert Infix to Postfix (Shunting-yard algorithm) ---
    postfix = []
    operator_stack = []

    # Simple tokenization by iterating through the string
    # A more robust solution would use regex or a dedicated lexer
    tokens = [char for char in infix_expression if not char.isspace()]

    for token in tokens:
        if token.isalnum():  # Token is an operand (variable or number)
            postfix.append(token)
        elif token == '(':
            operator_stack.append(token)
        elif token == ')':
            # Pop operators until an opening parenthesis is found
            while operator_stack and operator_stack[-1] != '(':
                postfix.append(operator_stack.pop())
            operator_stack.pop()  # Discard the '('
        else:  # Token is an operator
            while (operator_stack and
                   operator_stack[-1] != '(' and
                   precedence.get(operator_stack[-1], 0) >= precedence.get(token, 0)):
                postfix.append(operator_stack.pop())
            operator_stack.append(token)

    # Pop any remaining operators from the stack
    while operator_stack:
        postfix.append(operator_stack.pop())

    print(f"\n📝 Intermediate Step: Postfix notation is -> {' '.join(postfix)}")

    # --- Step 2: Generate Assembly from Postfix ---
    assembly_code = []
    for token in postfix:
        if token in op_to_instruction:
            # It's an operator, so add the corresponding instruction
            assembly_code.append(op_to_instruction[token])
        else:
            # It's an operand, so push it onto the stack
            assembly_code.append(f"PUSH {token}")

    return assembly_code

# --- Main Execution ---
if __name__ == "__main__":
    # Get input from the user
    input_expr = input("Enter an arithmetic expression (e.g., (a+b)*c): ")

    if not input_expr:
        print("No expression entered. Exiting.")
    else:
        print(f"\n--- Input Expression ---")
        print(input_expr)
        print("-" * 25)

        # Generate and print the assembly code
        generated_code = generate_stack_code(input_expr)

        print("\n--- Generated Assembly Code ---")
        for line in generated_code:
            print(line)
        print("-" * 25)

Enter an arithmetic expression (e.g., (a+b)*c): (a+b)*c

--- Input Expression ---
(a+b)*c
-------------------------

📝 Intermediate Step: Postfix notation is -> a b + c *

--- Generated Assembly Code ---
PUSH a
PUSH b
ADD
PUSH c
MUL
-------------------------
